# AML Project 1 — Missing Label Learning with Logistic Lasso Regression

**Advanced Machine Learning · 2026**

This notebook walks through all three tasks of the project:

| Task | Description |
|------|-------------|
| Task 1 | Data preparation and missing label generation |
| Task 2 | FISTA implementation and comparison with sklearn |
| Task 3 | UnlabeledLogReg — Naive / LabelProp / EM / Oracle comparison |

**Required files in the same directory:**
```
preprocessing.py
missing_data.py
fista.py
unlabeled_logreg.py
one_dollar_spin_and_go.csv   ← raw data
```

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import roc_auc_score, f1_score

from preprocessing import run_pipeline
from missing_data import generate_missing
from fista import LogisticRegressionFISTA, FISTASelector
from unlabeled_logreg import UnlabeledLogReg

SEED = 42
np.random.seed(SEED)

print('All imports successful.')

---
## Task 1 — Data Preparation and Missing Label Generation

### 1.1 Preprocessing Pipeline

The pipeline applies four steps in sequence:
1. **Column selection** — drop non-informative columns
2. **Target encoding** — map four outcomes to binary 0/1
3. **Log transform** — `log(1+x)` on skewed monetary features
4. **Min-Max normalisation** — scale all features to [0, 1]

In [ ]:
INPUT_FILE  = 'one_dollar_spin_and_go.csv'
OUTPUT_FILE = 'poker_data_preprocessed.csv'

run_pipeline(INPUT_FILE, OUTPUT_FILE)
print(f'Preprocessed data saved to {OUTPUT_FILE}')

### 1.2 Load and Inspect Dataset

In [ ]:
df = pd.read_csv(OUTPUT_FILE)

print(f'Shape: {df.shape}')
print(f'Positive class rate: {df["result"].mean():.3f}')
print(f'\nFeature columns: {df.drop(columns=["result"]).columns.tolist()}')
df.head()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
feature_cols = df.drop(columns=['result']).columns

for ax, col in zip(axes.flatten(), feature_cols):
    df[col].hist(bins=40, ax=ax, color='#378ADD', alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
    ax.grid(False)

fig.suptitle('Feature Distributions (after preprocessing)', fontsize=14)
plt.tight_layout()
plt.show()

### 1.3 Train / Validation / Test Split

In [ ]:
X = df.drop(columns=['result']).to_numpy()
y = df['result'].to_numpy()

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEED, stratify=y_trainval
)

print(f'Train:      {X_train.shape[0]:>6} samples  ({y_train.mean():.3f} positive)')
print(f'Validation: {X_valid.shape[0]:>6} samples  ({y_valid.mean():.3f} positive)')
print(f'Test:       {X_test.shape[0]:>6} samples  ({y_test.mean():.3f} positive)')

### 1.4 Missing Label Generation Schemes

Four mechanisms are implemented in `missing_data.py`. All return `y_obs` where missing labels are replaced by `-1`.

| Scheme | `P(S=1 \| ...)` | Correctable? |
|--------|-----------------|-------------|
| MCAR   | constant `c` | Yes |
| MAR1   | single feature `X_j` | Yes |
| MAR2   | all features via random projection | Yes |
| MNAR   | feature + true label `Y` | No |

In [ ]:
C          = 0.3
FEAT_IDX   = 11   # pot_growth — only feature that can drive missingness to target c
Y_WEIGHT   = 5.0  # MNAR label influence

schemes = {
    'MCAR': {'scheme': 'mcar'},
    'MAR1': {'scheme': 'mar1', 'feature_idx': FEAT_IDX},
    'MAR2': {'scheme': 'mar2'},
    'MNAR': {'scheme': 'mnar', 'feature_idx': FEAT_IDX, 'y_weight': Y_WEIGHT},
}

print(f'Target missing rate: c = {C}\n')
print(f'{"Scheme":<8} {"Missing":>10} {"Rate":>8}')
print('-' * 30)

for name, kwargs in schemes.items():
    y_obs = generate_missing(X_train, y_train, c=C, random_state=SEED, **kwargs)
    n_miss = (y_obs == -1).sum()
    print(f'{name:<8} {n_miss:>10}   {n_miss/len(y_obs)*100:>6.1f}%')

In [ ]:
# Visualise missingness pattern for each scheme
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

feature_names = df.drop(columns=['result']).columns.tolist()

for ax, (name, kwargs) in zip(axes, schemes.items()):
    y_obs = generate_missing(X_train, y_train, c=C, random_state=SEED, **kwargs)
    missing_mask = (y_obs == -1)

    # Compare pot_growth distribution for missing vs observed
    ax.hist(X_train[~missing_mask, FEAT_IDX], bins=30, alpha=0.6,
            color='#378ADD', label='Observed', density=True)
    ax.hist(X_train[missing_mask, FEAT_IDX], bins=30, alpha=0.6,
            color='#E24B4A', label='Missing', density=True)
    ax.set_title(name)
    ax.set_xlabel('pot_growth (feature 11)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('pot_growth Distribution: Observed vs Missing Labels', fontsize=13)
plt.tight_layout()
plt.show()

### 1.5 Generate and Save Missing-Label Datasets

In [ ]:
for name, kwargs in schemes.items():
    y_obs = generate_missing(X_train, y_train, c=C, random_state=SEED, **kwargs)
    df_out = pd.DataFrame(X_train, columns=feature_names)
    df_out['y_obs'] = y_obs
    fname = f'poker_{name.lower()}.csv'
    df_out.to_csv(fname, index=False)
    print(f'{fname} saved — {(y_obs==-1).sum()} missing labels')

---
## Task 2 — FISTA Implementation

### 2.1 Mathematical Background

FISTA minimises the Logistic Lasso objective:

$$\min_{\beta} \underbrace{\frac{1}{n} \sum_{i=1}^{n} \log(1 + e^{-y_i x_i^\top \beta})}_{\text{logistic loss}} + \underbrace{\lambda \|\beta\|_1}_{\text{L1 penalty}}$$

**Each iteration:**
1. Compute gradient at momentum point: $\nabla \mathcal{L}(z_k) = \frac{1}{n} X^\top (\sigma(Xz_k) - y)$
2. Gradient step: $v = z_k - \frac{1}{L} \nabla \mathcal{L}(z_k)$
3. Soft thresholding: $w_{k+1} = \text{sign}(v) \cdot \max(|v| - \lambda/L, 0)$
4. Nesterov momentum: $t_{k+1} = \frac{1 + \sqrt{1 + 4t_k^2}}{2}$, $z_{k+1} = w_{k+1} + \frac{t_k - 1}{t_{k+1}}(w_{k+1} - w_k)$

where the Lipschitz constant $L = \|X\|_F^2 / (4n)$.

### 2.2 Single Lambda — Quick Test

In [ ]:
model_single = LogisticRegressionFISTA(lambda_val=0.01, max_iter=1000, tol=1e-5)
model_single.fit(X_train, y_train)

print(f'Coefficients (incl. bias): {len(model_single.w)}')
print(f'Zero coefficients:         {(model_single.w[1:] == 0).sum()} / {X_train.shape[1]}')
print()

for metric in ['recall', 'precision', 'f1', 'balanced_accuracy', 'roc_auc', 'pr_auc']:
    score = model_single.validate(X_test, y_test, metric)
    print(f'  {metric:<22}: {score:.4f}')

### 2.3 Lambda Selection via Validation Set

In [ ]:
LAMBDAS = np.logspace(-4, 1, 30)
MEASURE = 'f1'

selector = FISTASelector(lambdas=LAMBDAS, max_iter=1000, tol=1e-5)
selector.fit(X_train, y_train, X_valid, y_valid, measure=MEASURE)

print(f'Best lambda:        {selector.best_lambda:.6f}')
print(f'Validation {MEASURE}: {selector.scores[selector.best_lambda]:.4f}')

### 2.4 Validation Metric vs Lambda

In [ ]:
selector.plot(measure=MEASURE)

### 2.5 Regularisation Path

In [ ]:
selector.plot_coefficients()

### 2.6 Test Set Evaluation — FISTA

In [ ]:
best = selector.best_model

print('FISTA — Test Set')
print('-' * 35)
for metric in ['recall', 'precision', 'f1', 'balanced_accuracy', 'roc_auc', 'pr_auc']:
    score = best.validate(X_test, y_test, metric)
    print(f'  {metric:<22}: {score:.4f}')

w = best.w[1:]
print(f'\nZero coefficients: {(w == 0).sum()} / {len(w)}')

### 2.7 FISTA vs sklearn Comparison

In [ ]:
C_sklearn = 1.0 / (selector.best_lambda * len(y_train))

sk_model = SklearnLR(
    C=C_sklearn, solver='saga',
    l1_ratio=1.0, max_iter=2000, random_state=SEED
)
sk_model.fit(X_train, y_train)
sk_proba = sk_model.predict_proba(X_test)[:, 1]
sk_pred  = (sk_proba >= 0.5).astype(int)

fista_proba = best.predict_proba(X_test)
fista_pred  = (fista_proba >= 0.5).astype(int)

print(f'{"Metric":<22}  {"FISTA":>8}  {"sklearn":>8}')
print('-' * 42)
print(f'{"F1 score":<22}  {f1_score(y_test, fista_pred):>8.4f}  {f1_score(y_test, sk_pred):>8.4f}')
print(f'{"ROC AUC":<22}  {roc_auc_score(y_test, fista_proba):>8.4f}  {roc_auc_score(y_test, sk_proba):>8.4f}')
print(f'{"Zero coefficients":<22}  {(best.w[1:]==0).sum():>8}  {(sk_model.coef_[0]==0).sum():>8}')

In [ ]:
# Coefficient comparison bar chart
fista_coefs = best.w[1:]
sk_coefs    = sk_model.coef_[0]

x = np.arange(len(feature_names))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, fista_coefs, width, label='FISTA', color='#378ADD', alpha=0.85)
ax.bar(x + width/2, sk_coefs,    width, label='sklearn', color='#E24B4A', alpha=0.85)
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Coefficient value')
ax.set_title('FISTA vs sklearn — Coefficient Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Task 3 — Unlabeled Logistic Regression

### 3.1 Class Overview

`UnlabeledLogReg` encapsulates all Task 3 logic:

| Method | Description |
|--------|-------------|
| `fit(X, y_obs, X_valid, y_valid)` | Train with labeled + unlabeled data |
| `naive_fit(X, y_obs, X_valid, y_valid)` | Train on labeled only (Naive baseline) |
| `oracle_fit(X, y_true, X_valid, y_valid)` | Train on all true labels (Oracle benchmark) |
| `compare(X_test, y_test)` | Print accuracy / balanced acc. / F1 / ROC AUC |
| `run_schemes(...)` | MCAR / MAR1 / MAR2 / MNAR comparison + plot |
| `run_mcar_sensitivity(...)` | Performance vs missing rate c + plot |

Two Y-completion algorithms:
- **Label Propagation** — one-shot: train on labeled → predict unlabeled → retrain
- **EM** — iterative: repeat (M-step: train, E-step: re-predict unlabeled) until convergence

### 3.2 Single Scheme Demo — MCAR

In [ ]:
# Generate missing labels for demo
y_obs_demo = generate_missing(X_train, y_train, scheme='mcar', c=0.3, random_state=SEED)
print(f'Missing labels: {(y_obs_demo==-1).sum()} / {len(y_obs_demo)} ({(y_obs_demo==-1).mean()*100:.1f}%)')

In [ ]:
model_em = UnlabeledLogReg(
    method='em',
    lambdas=np.logspace(-4, 1, 15),
    measure='f1',
    max_iter_em=10,
    max_iter_fista=500,
    feature_idx=11,
    mnar_y_weight=5.0,
    random_state=SEED,
)

# Train all three variants
model_em.naive_fit(X_train, y_obs_demo, X_valid, y_valid)
model_em.fit(X_train, y_obs_demo, X_valid, y_valid)
model_em.oracle_fit(X_train, y_train, X_valid, y_valid)

# Compare on test set
results_demo = model_em.compare(X_test, y_test)

### 3.3 Experiment 1 — Four Missing Data Schemes (c = 0.3)

Compares Naive / LabelProp / Oracle and Naive / EM / Oracle under each scheme.

In [ ]:
model_lp = UnlabeledLogReg(
    method='label_propagation',
    lambdas=np.logspace(-4, 1, 15),
    measure='f1',
    max_iter_fista=500,
    feature_idx=11,
    mnar_y_weight=5.0,
    random_state=SEED,
)

results_lp = model_lp.run_schemes(
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    c=0.3,
    save_plot=True,
    plot_filename='results_schemes_labelprop.png',
)

In [ ]:
model_em2 = UnlabeledLogReg(
    method='em',
    lambdas=np.logspace(-4, 1, 15),
    measure='f1',
    max_iter_em=10,
    max_iter_fista=500,
    feature_idx=11,
    mnar_y_weight=5.0,
    random_state=SEED,
)

results_em = model_em2.run_schemes(
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    c=0.3,
    save_plot=True,
    plot_filename='results_schemes_em.png',
)

### 3.4 Experiment 2 — MCAR: Effect of Missing Rate c

In [ ]:
model_sens = UnlabeledLogReg(
    method='em',
    lambdas=np.logspace(-4, 1, 15),
    measure='f1',
    max_iter_em=10,
    max_iter_fista=500,
    feature_idx=11,
    mnar_y_weight=5.0,
    c_values=[0.1, 0.2, 0.3, 0.4, 0.5],
    random_state=SEED,
)

mcar_results = model_sens.run_mcar_sensitivity(
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    save_plot=True,
    plot_filename='results_mcar_c.png',
)

### 3.5 Summary Table — MNAR vs Oracle Gap

In [ ]:
# Build a combined summary across all schemes for EM results
schemes_list = ['mcar', 'mar1', 'mar2', 'mnar']
metrics_list = ['accuracy', 'balanced_accuracy', 'f1', 'roc_auc']
method_label = 'EM'

rows = []
for scheme in schemes_list:
    for method in ['Naive', method_label, 'Oracle']:
        if method not in results_em.get(scheme, {}):
            continue
        m = results_em[scheme][method]
        rows.append({
            'Scheme': scheme.upper(),
            'Method': method,
            'Accuracy': round(m['accuracy'], 4),
            'Bal. Acc.': round(m['balanced_accuracy'], 4),
            'F1': round(m['f1'], 4),
            'ROC AUC': round(m['roc_auc'], 4),
        })

summary_df = pd.DataFrame(rows)

# Highlight MNAR rows
def highlight_mnar(row):
    if row['Scheme'] == 'MNAR':
        return ['background-color: #FEF3C7'] * len(row)
    return [''] * len(row)

summary_df.style.apply(highlight_mnar, axis=1)

In [ ]:
# Oracle vs best method gap per scheme
print(f'{"Scheme":<8}  {"Best method":<12}  {"Best F1":>8}  {"Oracle F1":>10}  {"Gap":>8}')
print('-' * 55)

for scheme in schemes_list:
    if scheme not in results_em:
        continue
    r = results_em[scheme]
    oracle_f1 = r.get('Oracle', {}).get('f1', float('nan'))
    
    best_method, best_f1 = 'Naive', r.get('Naive', {}).get('f1', 0)
    for m in [method_label]:
        if m in r and r[m]['f1'] > best_f1:
            best_method, best_f1 = m, r[m]['f1']
    
    gap = oracle_f1 - best_f1
    print(f'{scheme.upper():<8}  {best_method:<12}  {best_f1:>8.4f}  {oracle_f1:>10.4f}  {gap:>8.4f}')

---
## Summary of Findings

**Task 1:**
- The poker dataset was preprocessed into 12 numerical features with log transformation and Min-Max normalisation.
- Four missing label mechanisms were implemented. Only `pot_growth` (feature index 11) can reliably drive missingness to the target rate due to the dataset's skewed distributions.

**Task 2:**
- Custom FISTA achieves identical F1 (0.8330) and ROC AUC (0.9436) to sklearn's L1 solver.
- The regularisation path shows `total_bet` as the strongest positive predictor and `pot_flop` as the strongest negative predictor.

**Task 3:**
- **MCAR / MAR1 / MAR2:** All methods perform near-identically and close to Oracle. With 61K samples and 30% missing, the Naive method already retains sufficient data.
- **MNAR:** All methods degrade significantly. EM performs worst due to **confirmation bias** — the initial biased predictions are reinforced iteratively.
- **c sensitivity (MCAR):** Performance is insensitive to the missing rate because the dataset is large enough that even 50% missing leaves ~30K labeled samples.